In [2]:
import pandas as pd

df = pd.read_csv('../data/creditcard.csv')

print(df.shape)
print(df['Class'].value_counts())
print(df['Class'].value_counts(normalize=True))

(284807, 31)
Class
0    284315
1       492
Name: count, dtype: int64
Class
0    0.998273
1    0.001727
Name: proportion, dtype: float64


In [3]:
from sklearn.model_selection import train_test_split

#X is all the input features, like transaction data
X = df.drop('Class', axis=1)
#y is the answer column we are trying to predict
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    #hold 20% for testing
    test_size=0.2,
    #keeps fraud ratio identical in both sets
    stratify=y,
    #makes the split reproducible because you get the same split each run
    random_state=42
)

#Check the split worked and the ratio is preserved
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
#since fraud is 1 and legit is 0, the avg of column is fraud proportion
print("\nFraud % in train:", y_train.mean())
print("Fraud % in test:", y_test.mean())


Train shape: (227845, 30)
Test shape: (56962, 30)

Fraud % in train: 0.001729245759178389
Fraud % in test: 0.0017204452090867595


In [4]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

scaler = StandardScaler()

#prevent data leakage which is where the test data influences the training
#fit_transform on train so it learns and applies
X_train_scaled = scaler.fit_transform(X_train)
#only transform on test
X_test_scaled = scaler.transform(X_test)

baseline = LogisticRegression(max_iter= 1000)
baseline.fit(X_train_scaled, y_train)

y_pred = baseline.predict(X_test_scaled)

In [6]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

#the impressive number
print("Accuracy:", accuracy_score(y_test, y_pred))

#honest picture
print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification report:")
print(classification_report(y_test, y_pred))



Accuracy: 0.9991397773954567

Confusion matrix:
[[56851    13]
 [   36    62]]

Classification report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.83      0.63      0.72        98

    accuracy                           1.00     56962
   macro avg       0.91      0.82      0.86     56962
weighted avg       1.00      1.00      1.00     56962



- from these results, we can see that we have retained a recall score of 0.63. this means that of all of the real fraud cases, the model caught 63% of them
- from the precision value (0.83), we can determine that out of the cases that the model flagged as fraud, 83% actually were.
- we need to focus on the recall and the precision-recall tradeoff, instead of the accuracy.

In [7]:
#same logistic regression but weighing the rare class more heavily

weighted = LogisticRegression(max_iter=1000, class_weight = 'balanced')
weighted.fit(X_train_scaled, y_train)

y_pred_weighted = weighted.predict(X_test_scaled)

print("Confusion matrix(weighted):")
print(confusion_matrix(y_test, y_pred_weighted))
print("\nClassification report (weighted):")
print(classification_report(y_test, y_pred_weighted))

Confusion matrix(weighted):
[[55478  1386]
 [    8    90]]

Classification report (weighted):
              precision    recall  f1-score   support

           0       1.00      0.98      0.99     56864
           1       0.06      0.92      0.11        98

    accuracy                           0.98     56962
   macro avg       0.53      0.95      0.55     56962
weighted avg       1.00      0.98      0.99     56962



I predicted that with the fraud class more heavily weighted, that the recall value would increase and the precision value would decrease
This was the case. 
Only 6% of the cases that the model flagged as fraud were actually fraud (precision).
Recall increased from 63% to 92%, so the model caught 92% of fraud cases.